In [42]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from pathlib import Path
from collections import defaultdict
import inspect

In [43]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.base import clone
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import label_binarize

In [44]:
DATA_DIR = Path("data/raw_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DATA_DIR = Path("data/final_data")
FINAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
ELO_PATH = DATA_DIR / "elo_ratings" / "eloratings.csv"

In [45]:
df = pd.read_csv(f"{DATA_DIR}/international_matches/results.csv")
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49477 entries, 0 to 49476
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49477 non-null  object 
 1   home_team   49477 non-null  object 
 2   away_team   49477 non-null  object 
 3   home_score  49453 non-null  float64
 4   away_score  49453 non-null  float64
 5   tournament  49477 non-null  object 
 6   city        49477 non-null  object 
 7   country     49477 non-null  object 
 8   neutral     49477 non-null  bool   
dtypes: bool(1), float64(2), object(6)
memory usage: 3.1+ MB


In [47]:
df.isna().sum()

date           0
home_team      0
away_team      0
home_score    24
away_score    24
tournament     0
city           0
country        0
neutral        0
dtype: int64

In [48]:
# What are these NA values ??
missing_score_matches = df[df["home_score"].isna() | df["away_score"].isna()].copy()
missing_score_matches.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49453,2026-06-24,Mexico,Czech Republic,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False
49454,2026-06-24,South Africa,South Korea,NaN,NaN,FIFA World Cup,Guadalupe,Mexico,True
49455,2026-06-24,Canada,Switzerland,NaN,NaN,FIFA World Cup,Vancouver,Canada,False
49456,2026-06-24,Bosnia and Herzegovina,Qatar,NaN,NaN,FIFA World Cup,Seattle,United States,True
49457,2026-06-24,Scotland,Brazil,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True


In [49]:
# Drop these future values because we don't need them
df = df.dropna(subset=["home_score", "away_score"]).copy()

In [50]:
# Make the actual dataframe from this now
matches = pd.DataFrame()

matches["match_id"] = range(1, len(df) + 1)
matches["date"] = pd.to_datetime(df["date"])
matches["team_a"] = df["home_team"].str.strip()
matches["team_b"] = df["away_team"].str.strip()
matches["team_a_score"] = df["home_score"].astype(int)
matches["team_b_score"] = df["away_score"].astype(int)
matches["tournament"] = df["tournament"].str.strip()
matches["city"] = df["city"].str.strip()
matches["country"] = df["country"].str.strip()
matches["neutral"] = df["neutral"].astype(bool)
matches["is_team_a_home"] = 1

In [51]:
matches.head()

,match_id,date,team_a,team_b,team_a_score,team_b_score,tournament,city,country,neutral,is_team_a_home
0,1,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,1
1,2,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,1
2,3,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,1
3,4,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,1
4,5,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,1


In [52]:
targets = []

for a_score, b_score in zip(matches["team_a_score"], matches["team_b_score"]):
    if a_score > b_score:
        targets.append(1)
    elif a_score < b_score:
        targets.append(2)
    else:
        targets.append(0)

matches["target"] = targets
matches[["target", "team_a_score", "team_b_score"]].head()

,target,team_a_score,team_b_score
0,0,0,0
1,1,4,2
2,1,2,1
3,0,2,2
4,1,3,0


In [53]:
matches["tournament_lower"]=matches["tournament"].str.lower()
matches["is_world_cup_qualifier"] = (
    matches["tournament_lower"].str.contains(
        r"fifa world cup.*qualification|fifa world cup.*qualifier",
        regex=True,
        na=False
    )
).astype(int)

matches["is_world_cup"] = (
    matches["tournament_lower"].eq("fifa world cup")
).astype(int)

In [54]:
matches["is_continental"] = (
    matches["tournament_lower"].str.contains(
        r"uefa euro|copa américa|copa america|african cup of nations|afc asian cup|concacaf gold cup|ofc nations cup",
        regex=True,
        na=False
    )
).astype(int)

matches["is_friendly"] = (
    matches["tournament_lower"].str.contains(
        r"friendly",
        regex=True,
        na=False
    )
).astype(int)

In [55]:
matches[
    [
        "is_world_cup",
        "is_world_cup_qualifier",
        "is_continental",
        "is_friendly"
    ]
].sum()

is_world_cup               1012
is_world_cup_qualifier     8771
is_continental             8511
is_friendly               18388
dtype: int64

In [56]:
matches.head()

,match_id,date,team_a,team_b,team_a_score,team_b_score,tournament,city,country,neutral,is_team_a_home,target,tournament_lower,is_world_cup_qualifier,is_world_cup,is_continental,is_friendly
0,1,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,1,0,friendly,0,0,0,1
1,2,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,1,1,friendly,0,0,0,1
2,3,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,1,1,friendly,0,0,0,1
3,4,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,1,0,friendly,0,0,0,1
4,5,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,1,1,friendly,0,0,0,1


In [57]:
team_a_history = matches[["team_a","team_b", "team_a_score", "team_b_score", "target", "match_id", "date"]].copy()
team_b_history = matches[["team_a","team_b", "team_a_score", "team_b_score", "target", "match_id", "date"]].copy()

team_a_history = team_a_history.rename(
    columns = {
        "team_a":"team",
        "team_b":"opponent",
        "team_a_score":"goals_for",
        "team_b_score":"goals_against"
    }
)

team_b_history = team_b_history.rename(
    columns = {
        "team_a":"opponent",
        "team_b":"team",
        "team_a_score":"goals_against",
        "team_b_score":"goals_for"
    }
)

In [58]:
def get_points_for_team(match_row, team_name):
    if match_row["team_a"] == team_name:
        goals_for = match_row["team_a_score"]
        goals_against = match_row["team_b_score"]
    else:
        goals_for = match_row["team_b_score"]
        goals_against = match_row["team_a_score"]

    if goals_for > goals_against:
        return 3
    elif goals_for == goals_against:
        return 1
    else:
        return 0

In [59]:
team_a_history = matches[
    ["match_id", "date", "team_a", "team_b", "team_a_score", "team_b_score"]
].copy()

team_a_history = team_a_history.rename(
    columns={
        "team_a": "team",
        "team_b": "opponent",
        "team_a_score": "goals_for",
        "team_b_score": "goals_against",
    }
)

team_b_history = matches[
    ["match_id", "date", "team_a", "team_b", "team_a_score", "team_b_score"]
].copy()

team_b_history = team_b_history.rename(
    columns={
        "team_b": "team",
        "team_a": "opponent",
        "team_b_score": "goals_for",
        "team_a_score": "goals_against",
    }
)

team_history = pd.concat([team_a_history, team_b_history], ignore_index=True)

team_history["points"] = np.where(
    team_history["goals_for"] > team_history["goals_against"], 3,
    np.where(
        team_history["goals_for"] == team_history["goals_against"], 1,
        0
    )
)

team_history = team_history.sort_values(["team", "date", "match_id"]).reset_index(drop=True)

team_history["points_last_5"] = (
    team_history
    .groupby("team")["points"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).sum())
)

In [60]:
team_a_points = team_history[
    ["match_id", "team", "points_last_5"]
].rename(
    columns={
        "team": "team_a",
        "points_last_5": "team_a_points_last_5"
    }
)

team_b_points = team_history[
    ["match_id", "team", "points_last_5"]
].rename(
    columns={
        "team": "team_b",
        "points_last_5": "team_b_points_last_5"
    }
)

matches = matches.merge(
    team_a_points,
    on=["match_id", "team_a"],
    how="left"
)

matches = matches.merge(
    team_b_points,
    on=["match_id", "team_b"],
    how="left"
)

matches["points_last_5_diff"] = (
    matches["team_a_points_last_5"] - matches["team_b_points_last_5"]
)

In [61]:
matches[
    [
        "date",
        "team_a",
        "team_b",
        "team_a_points_last_5",
        "team_b_points_last_5",
        "points_last_5_diff"
    ]
].head()

,date,team_a,team_b,team_a_points_last_5,team_b_points_last_5,points_last_5_diff
0,1872-11-30,Scotland,England,NaN,NaN,NaN
1,1873-03-08,England,Scotland,1.0,1.0,0.0
2,1874-03-07,Scotland,England,1.0,4.0,-3.0
3,1875-03-06,England,Scotland,4.0,4.0,0.0
4,1876-03-04,Scotland,England,5.0,5.0,0.0


In [62]:
def add_pre_match_elo(matches_df, elo_df, team_col, new_col):
    output_parts = []

    temp = matches_df[["match_id", "date", team_col]].copy()
    temp = temp.rename(columns={team_col: "team"})

    for team_name, team_matches in temp.groupby("team"):
        team_elo = elo_df[elo_df["team"] == team_name][["date", "rating"]].copy()

        if team_elo.empty:
            team_matches[new_col] = pd.NA
        else:
            team_matches = team_matches.sort_values("date")
            team_elo = team_elo.sort_values("date")

            merged = pd.merge_asof(
                team_matches,
                team_elo,
                on="date",
                direction="backward",
                allow_exact_matches=False
            )

            team_matches[new_col] = merged["rating"]

        output_parts.append(team_matches[["match_id", new_col]])

    return pd.concat(output_parts, ignore_index=True)

In [63]:
elo = pd.read_csv(ELO_PATH)

elo["date"] = pd.to_datetime(elo["date"], format="mixed", errors="coerce")
elo["team"] = elo["team"].str.strip()
elo["rating"] = pd.to_numeric(elo["rating"], errors="coerce")

matches["date"] = pd.to_datetime(matches["date"])
matches["team_a"] = matches["team_a"].str.strip()
matches["team_b"] = matches["team_b"].str.strip()

elo = elo.sort_values(["team", "date"]).reset_index(drop=True)
matches = matches.sort_values(["date", "match_id"]).reset_index(drop=True)

team_a_elo = add_pre_match_elo(matches, elo, "team_a", "team_a_elo")
team_b_elo = add_pre_match_elo(matches, elo, "team_b", "team_b_elo")

matches = matches.merge(team_a_elo, on="match_id", how="left")
matches = matches.merge(team_b_elo, on="match_id", how="left")

matches["elo_diff"] = matches["team_a_elo"] - matches["team_b_elo"]

C:\Users\SAMARTH\AppData\Local\Temp\ipykernel_28356\3641760760.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(output_parts, ignore_index=True)
C:\Users\SAMARTH\AppData\Local\Temp\ipykernel_28356\3641760760.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(output_parts, ignore_index=True)


In [64]:
# Basic pi ratings
matches = matches.sort_values(["date", "match_id"]).reset_index(drop=True).copy()

INITIAL_PI = 0.0
LEARNING_RATE = 0.05
HOME_ADVANTAGE = 0.25
MAX_GOAL_DIFF = 5

team_pi = defaultdict(lambda: INITIAL_PI)

team_a_pi_list = []
team_b_pi_list = []
pi_diff_list = []
expected_gd_list = []
actual_gd_list = []
pi_error_list = []

for _, row in matches.iterrows():
    team_a = row["team_a"]
    team_b = row["team_b"]

    team_a_pi = team_pi[team_a]
    team_b_pi = team_pi[team_b]

    home_advantage = 0 if row["neutral"] else HOME_ADVANTAGE

    expected_gd = team_a_pi - team_b_pi + home_advantage

    actual_gd = row["team_a_score"] - row["team_b_score"]
    actual_gd = np.clip(actual_gd, -MAX_GOAL_DIFF, MAX_GOAL_DIFF)

    error = actual_gd - expected_gd

    team_a_pi_list.append(team_a_pi)
    team_b_pi_list.append(team_b_pi)
    pi_diff_list.append(team_a_pi - team_b_pi)
    expected_gd_list.append(expected_gd)
    actual_gd_list.append(actual_gd)
    pi_error_list.append(error)

    update = LEARNING_RATE * error

    team_pi[team_a] += update
    team_pi[team_b] -= update

matches["team_a_pi"] = team_a_pi_list
matches["team_b_pi"] = team_b_pi_list
matches["pi_diff"] = pi_diff_list

matches["pi_expected_goal_diff"] = expected_gd_list
matches["pi_actual_goal_diff"] = actual_gd_list
matches["pi_error"] = pi_error_list

In [65]:
matches.tail()

,match_id,date,team_a,team_b,team_a_score,team_b_score,tournament,city,country,neutral,...,points_last_5_diff,team_a_elo,team_b_elo,elo_diff,team_a_pi,team_b_pi,pi_diff,pi_expected_goal_diff,pi_actual_goal_diff,pi_error
49448,49449,2026-06-22,Jordan,Algeria,1,2,FIFA World Cup,Santa Clara,United States,True,...,-8.0,NaN,NaN,NaN,1.219519,2.087010,-0.867491,-0.867491,-1,-0.132509
49449,49450,2026-06-23,Portugal,Uzbekistan,5,0,FIFA World Cup,Houston,United States,True,...,7.0,NaN,NaN,NaN,2.760579,1.449828,1.310752,1.310752,5,3.689248
49450,49451,2026-06-23,Colombia,DR Congo,1,0,FIFA World Cup,Zapopan,Mexico,True,...,1.0,NaN,NaN,NaN,3.005946,1.288544,1.717402,1.717402,1,-0.717402
49451,49452,2026-06-23,England,Ghana,0,0,FIFA World Cup,Foxborough,United States,True,...,6.0,NaN,NaN,NaN,2.988931,0.984624,2.004306,2.004306,0,-2.004306
49452,49453,2026-06-23,Panama,Croatia,0,1,FIFA World Cup,Toronto,Canada,True,...,1.0,NaN,NaN,NaN,1.282684,2.194090,-0.911405,-0.911405,-1,-0.088595


In [66]:
feature_cols = [
    "neutral",
    "is_team_a_home",
    "is_friendly",
    "is_world_cup",
    "is_world_cup_qualifier",
    "is_continental",

    "team_a_points_last_5",
    "team_b_points_last_5",
    "points_last_5_diff",

    "team_a_elo",
    "team_b_elo",
    "elo_diff",

    "team_a_pi",
    "team_b_pi",
    "pi_diff",
]

In [67]:
model_df = matches[feature_cols + ["target", "date"]].copy()
model_df = model_df.dropna(subset=["target"]).copy()
model_df["date"] = pd.to_datetime(model_df["date"])
model_df = model_df.sort_values("date").reset_index(drop=True)

In [68]:
model_df["year"] = model_df["date"].dt.year

print("Date range:")
print(model_df["date"].min(), "to", model_df["date"].max())

print("\nRows per year:")
display(model_df["year"].value_counts().sort_index())

print("\nTarget distribution overall:")
display(model_df["target"].value_counts(normalize=True).sort_index())

print("\nTarget distribution by year:")
display(
    pd.crosstab(
        model_df["year"],
        model_df["target"],
        normalize="index"
    ).round(3)
)

print("\nNumber of 2026 matches:")
print((model_df["year"] == 2026).sum())

Date range:
1872-11-30 00:00:00 to 2026-06-23 00:00:00

Rows per year:


year
1872       1
1873       1
1874       1
1875       1
1876       2
        ... 
2022     970
2023    1054
2024    1231
2025    1002
2026     359
Name: count, Length: 155, dtype: int64


Target distribution overall:


target
0    0.227428
1    0.490102
2    0.282470
Name: proportion, dtype: float64


Target distribution by year:


target,0,1,2
year,,,
1872,1.000,0.000,0.000
1873,0.000,1.000,0.000
1874,0.000,1.000,0.000
1875,1.000,0.000,0.000
1876,0.000,1.000,0.000
...,...,...,...
2022,0.227,0.498,0.275
2023,0.213,0.467,0.320
2024,0.249,0.461,0.289



Number of 2026 matches:
359


In [69]:
model_df.to_csv(FINAL_DATA_DIR / "model_df.csv", index=False)

In [70]:
DATE_COL = "date"
TARGET_COL = "target"
CLASS_ORDER = [0, 1, 2]

TRAIN_END_2023 = pd.Timestamp("2023-12-31")
CV_START = pd.Timestamp("2024-01-01")
CV_END = pd.Timestamp("2025-12-31")
TEST_START_2026 = pd.Timestamp("2026-01-01")
TRAIN_END_2025 = pd.Timestamp("2025-12-31")

model_df = model_df.sort_values(DATE_COL).reset_index(drop=True).copy()

train_base_df = model_df[model_df[DATE_COL] <= TRAIN_END_2023].copy()

cv_df = model_df[
    (model_df[DATE_COL] >= CV_START) &
    (model_df[DATE_COL] <= CV_END)
].copy()

test_2026_df = model_df[model_df[DATE_COL] >= TEST_START_2026].copy()

train_to_2025_df = model_df[model_df[DATE_COL] <= TRAIN_END_2025].copy()

print("Date-based split sizes")
print("-" * 30)
print("Train base <= 2023:", len(train_base_df))
print("CV period 2024-2025:", len(cv_df))
print("Final test 2026:", len(test_2026_df))
print("Train final <= 2025:", len(train_to_2025_df))

print("\n2026 target distribution:")
display(test_2026_df[TARGET_COL].value_counts(normalize=True).reindex(CLASS_ORDER, fill_value=0))

Date-based split sizes
------------------------------
Train base <= 2023: 46861
CV period 2024-2025: 2233
Final test 2026: 359
Train final <= 2025: 49094

2026 target distribution:


target
0    0.253482
1    0.498607
2    0.247911
Name: proportion, dtype: float64

In [71]:
EPS = 1e-15
def get_X_y(df):
    X = df[feature_cols].copy()
    y = df[TARGET_COL].astype(int).copy()
    return X, y

def normalise_proba(y_proba):
    y_proba = np.asarray(y_proba, dtype=float)
    y_proba = np.clip(y_proba, EPS, 1.0)
    y_proba = y_proba / y_proba.sum(axis=1, keepdims=True)
    return y_proba

def align_proba_to_class_order(y_proba, model_classes, class_order=CLASS_ORDER):
    y_proba = np.asarray(y_proba, dtype=float)
    model_classes = list(model_classes)

    aligned = np.zeros((len(y_proba), len(class_order)))

    for output_idx, class_label in enumerate(class_order):
        if class_label in model_classes:
            input_idx = model_classes.index(class_label)
            aligned[:, output_idx] = y_proba[:, input_idx]

    return normalise_proba(aligned)

def multiclass_brier_score(y_true, y_proba, class_order=CLASS_ORDER):
    y_true_onehot = label_binarize(y_true, classes=class_order)
    y_proba = normalise_proba(y_proba)

    return np.mean(np.sum((y_proba - y_true_onehot) ** 2, axis=1))

def ranked_probability_score(y_true, y_proba, class_order=CLASS_ORDER):
    y_true_onehot = label_binarize(y_true, classes=class_order)
    y_proba = normalise_proba(y_proba)

    cumulative_proba = np.cumsum(y_proba, axis=1)
    cumulative_true = np.cumsum(y_true_onehot, axis=1)

    return np.mean(
        np.sum((cumulative_proba - cumulative_true) ** 2, axis=1)
        / (len(class_order) - 1)
    )

def evaluate_predictions(y_true, y_pred, y_proba, metadata=None):
    if metadata is None:
        metadata = {}

    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int).ravel()
    y_proba = normalise_proba(y_proba)

    result = {
        **metadata,
        "n_matches": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "log_loss": log_loss(y_true, y_proba, labels=CLASS_ORDER),
        "brier": multiclass_brier_score(y_true, y_proba, CLASS_ORDER),
        "rps": ranked_probability_score(y_true, y_proba, CLASS_ORDER),
    }

    return result

def evaluate_model_on_df(model, eval_df, metadata=None):
    X_eval, y_eval = get_X_y(eval_df)

    y_pred = model.predict(X_eval)
    y_pred = np.asarray(y_pred).ravel().astype(int)

    y_proba = model.predict_proba(X_eval)
    model_classes = getattr(model, "classes_", CLASS_ORDER)
    y_proba = align_proba_to_class_order(y_proba, model_classes, CLASS_ORDER)

    return evaluate_predictions(y_eval, y_pred, y_proba, metadata)

In [72]:
# Baseline Model
def evaluate_majority_baseline(train_df, eval_df, metadata=None):
    if metadata is None:
        metadata = {}

    _, y_train = get_X_y(train_df)
    _, y_eval = get_X_y(eval_df)

    majority_class = y_train.mode()[0]

    train_class_probs = (
        y_train.value_counts(normalize=True)
        .reindex(CLASS_ORDER, fill_value=0)
        .values
    )

    y_pred = np.full(shape=len(y_eval), fill_value=majority_class)
    y_proba = np.tile(train_class_probs, (len(y_eval), 1))

    baseline_metadata = {
        **metadata,
        "model": "Majority baseline",
        "calibration": "none",
    }

    return evaluate_predictions(y_eval, y_pred, y_proba, baseline_metadata)

In [73]:
# All the models 
def make_model_registry():
    models = {
        "Logistic Regression": Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    solver="lbfgs"
                )),
            ]
        ),

        "Random Forest": Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("model", RandomForestClassifier(
                    n_estimators=300,
                    max_depth=8,
                    min_samples_leaf=20,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1,
                )),
            ]
        ),

        "XGBoost": Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("model", XGBClassifier(
                    objective="multi:softprob",
                    num_class=3,
                    n_estimators=300,
                    max_depth=4,
                    learning_rate=0.05,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    eval_metric="mlogloss",
                    random_state=42,
                    n_jobs=-1,
                )),
            ]
        ),

        "CatBoost": Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("model", CatBoostClassifier(
                    loss_function="MultiClass",
                    iterations=300,
                    depth=4,
                    learning_rate=0.05,
                    random_seed=42,
                    verbose=False,
                )),
            ]
        ),
    }

    return models


model_registry = make_model_registry()

list(model_registry.keys())

['Logistic Regression', 'Random Forest', 'XGBoost', 'CatBoost']

In [74]:
def make_prefit_calibrator(fitted_model, method):
    """
    Makes a calibrated classifier from an already fitted model.

    Uses FrozenEstimator when available.
    Falls back to cv='prefit' for older sklearn versions.
    """

    try:
        return CalibratedClassifierCV(
            estimator=FrozenEstimator(fitted_model),
            method=method
        )

    except Exception:
        signature = inspect.signature(CalibratedClassifierCV)

        if "estimator" in signature.parameters:
            return CalibratedClassifierCV(
                estimator=fitted_model,
                method=method,
                cv="prefit"
            )

        return CalibratedClassifierCV(
            base_estimator=fitted_model,
            method=method,
            cv="prefit"
        )


def has_all_classes(y, class_order=CLASS_ORDER):
    return set(class_order).issubset(set(pd.Series(y).astype(int).unique()))


def split_fit_calibration_window(train_df, calibration_fraction=0.20, min_calibration_size=100):
    """
    Splits a training window into:
    - fit_df: used to train the base model
    - calibration_df: used to calibrate probabilities

    The calibration slice is the most recent part of the training window.
    """

    train_df = train_df.sort_values(DATE_COL).reset_index(drop=True).copy()

    n = len(train_df)

    if n < min_calibration_size * 2:
        raise ValueError(f"Training window too small for calibration split: {n} rows")

    calibration_size = max(int(n * calibration_fraction), min_calibration_size)
    calibration_size = min(calibration_size, n // 2)

    fit_df = train_df.iloc[:-calibration_size].copy()
    calibration_df = train_df.iloc[-calibration_size:].copy()

    if not has_all_classes(fit_df[TARGET_COL]):
        raise ValueError("Fit split does not contain all target classes")

    if not has_all_classes(calibration_df[TARGET_COL]):
        raise ValueError("Calibration split does not contain all target classes")

    return fit_df, calibration_df

In [75]:
def get_period(df, start=None, end=None):
    output = df.copy()

    if start is not None:
        output = output[output[DATE_COL] >= pd.Timestamp(start)]

    if end is not None:
        output = output[output[DATE_COL] <= pd.Timestamp(end)]

    return output.copy()


def add_fold(folds, cv_method, fold_name, train_start, train_end, valid_start, valid_end):
    train_df_fold = get_period(model_df, train_start, train_end)
    valid_df_fold = get_period(model_df, valid_start, valid_end)

    folds.append({
        "cv_method": cv_method,
        "fold": fold_name,
        "train_start": train_start,
        "train_end": train_end,
        "valid_start": valid_start,
        "valid_end": valid_end,
        "train_rows": len(train_df_fold),
        "valid_rows": len(valid_df_fold),
    })


cv_folds = []

# 1. Year-based CV
add_fold(
    cv_folds,
    cv_method="year_based",
    fold_name="validate_2024",
    train_start=None,
    train_end="2023-12-31",
    valid_start="2024-01-01",
    valid_end="2024-12-31",
)

add_fold(
    cv_folds,
    cv_method="year_based",
    fold_name="validate_2025",
    train_start=None,
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2025-12-31",
)


# 2. Expanding-window CV
expanding_windows = [
    ("validate_2024_H1", "2024-01-01", "2024-06-30"),
    ("validate_2024_H2", "2024-07-01", "2024-12-31"),
    ("validate_2025_H1", "2025-01-01", "2025-06-30"),
    ("validate_2025_H2", "2025-07-01", "2025-12-31"),
]

for fold_name, valid_start, valid_end in expanding_windows:
    train_end = (pd.Timestamp(valid_start) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    add_fold(
        cv_folds,
        cv_method="expanding_window",
        fold_name=fold_name,
        train_start=None,
        train_end=train_end,
        valid_start=valid_start,
        valid_end=valid_end,
    )


# 3. Rolling-window CV
# Uses the previous 5 years before each validation period.
rolling_windows = [
    ("validate_2024_H1", "2024-01-01", "2024-06-30"),
    ("validate_2024_H2", "2024-07-01", "2024-12-31"),
    ("validate_2025_H1", "2025-01-01", "2025-06-30"),
    ("validate_2025_H2", "2025-07-01", "2025-12-31"),
]

for fold_name, valid_start, valid_end in rolling_windows:
    valid_start_ts = pd.Timestamp(valid_start)

    train_start = (valid_start_ts - pd.DateOffset(years=5)).strftime("%Y-%m-%d")
    train_end = (valid_start_ts - pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    add_fold(
        cv_folds,
        cv_method="rolling_5_year_window",
        fold_name=fold_name,
        train_start=train_start,
        train_end=train_end,
        valid_start=valid_start,
        valid_end=valid_end,
    )


cv_folds_df = pd.DataFrame(cv_folds)

display(cv_folds_df)

,cv_method,fold,train_start,train_end,valid_start,valid_end,train_rows,valid_rows
0,year_based,validate_2024,None,2023-12-31,2024-01-01,2024-12-31,46861,1231
1,year_based,validate_2025,None,2024-12-31,2025-01-01,2025-12-31,48092,1002
2,expanding_window,validate_2024_H1,None,2023-12-31,2024-01-01,2024-06-30,46861,588
3,expanding_window,validate_2024_H2,None,2024-06-30,2024-07-01,2024-12-31,47449,643
4,expanding_window,validate_2025_H1,None,2024-12-31,2025-01-01,2025-06-30,48092,407
5,expanding_window,validate_2025_H2,None,2025-06-30,2025-07-01,2025-12-31,48499,595
6,rolling_5_year_window,validate_2024_H1,2019-01-01,2023-12-31,2024-01-01,2024-06-30,4635,588
7,rolling_5_year_window,validate_2024_H2,2019-07-01,2024-06-30,2024-07-01,2024-12-31,4737,643
8,rolling_5_year_window,validate_2025_H1,2020-01-01,2024-12-31,2025-01-01,2025-06-30,4717,407
9,rolling_5_year_window,validate_2025_H2,2020-07-01,2025-06-30,2025-07-01,2025-12-31,5117,595


In [76]:
cv_results = []

for fold_info in cv_folds:
    cv_method = fold_info["cv_method"]
    fold_name = fold_info["fold"]

    train_window_df = get_period(
        model_df,
        start=fold_info["train_start"],
        end=fold_info["train_end"]
    )

    valid_window_df = get_period(
        model_df,
        start=fold_info["valid_start"],
        end=fold_info["valid_end"]
    )

    if len(train_window_df) == 0 or len(valid_window_df) == 0:
        print(f"Skipping empty fold: {cv_method} - {fold_name}")
        continue

    try:
        fit_df, calibration_df = split_fit_calibration_window(train_window_df)

    except Exception as e:
        print(f"Skipping fold due to calibration split issue: {cv_method} - {fold_name}")
        print("Reason:", e)
        continue

    X_fit, y_fit = get_X_y(fit_df)
    X_calibration, y_calibration = get_X_y(calibration_df)

    # Baseline uses the whole training window.
    baseline_metadata = {
        "experiment": "cv_2024_2025",
        "cv_method": cv_method,
        "fold": fold_name,
        "train_start": fold_info["train_start"],
        "train_end": fold_info["train_end"],
        "valid_start": fold_info["valid_start"],
        "valid_end": fold_info["valid_end"],
        "train_rows": len(train_window_df),
        "fit_rows": len(fit_df),
        "calibration_rows": len(calibration_df),
        "valid_rows": len(valid_window_df),
    }

    cv_results.append(
        evaluate_majority_baseline(
            train_window_df,
            valid_window_df,
            metadata=baseline_metadata
        )
    )

    for model_name, base_estimator in model_registry.items():
        print(f"Running {cv_method} | {fold_name} | {model_name}")

        try:
            fitted_model = clone(base_estimator)
            fitted_model.fit(X_fit, y_fit)

            common_metadata = {
                "experiment": "cv_2024_2025",
                "cv_method": cv_method,
                "fold": fold_name,
                "train_start": fold_info["train_start"],
                "train_end": fold_info["train_end"],
                "valid_start": fold_info["valid_start"],
                "valid_end": fold_info["valid_end"],
                "train_rows": len(train_window_df),
                "fit_rows": len(fit_df),
                "calibration_rows": len(calibration_df),
                "valid_rows": len(valid_window_df),
                "model": model_name,
            }

            # Uncalibrated version
            cv_results.append(
                evaluate_model_on_df(
                    fitted_model,
                    valid_window_df,
                    metadata={
                        **common_metadata,
                        "calibration": "none",
                    }
                )
            )

            # Sigmoid and isotonic calibration
            for calibration_method in ["sigmoid", "isotonic"]:
                calibrated_model = make_prefit_calibrator(
                    fitted_model,
                    method=calibration_method
                )

                calibrated_model.fit(X_calibration, y_calibration)

                cv_results.append(
                    evaluate_model_on_df(
                        calibrated_model,
                        valid_window_df,
                        metadata={
                            **common_metadata,
                            "calibration": calibration_method,
                        }
                    )
                )

        except Exception as e:
            print(f"Error: {cv_method} | {fold_name} | {model_name}")
            print(e)


cv_results_df = pd.DataFrame(cv_results)

display(cv_results_df.head())
print("Total result rows:", len(cv_results_df))

Running year_based | validate_2024 | Logistic Regression
Running year_based | validate_2024 | Random Forest
Running year_based | validate_2024 | XGBoost
Running year_based | validate_2024 | CatBoost
Running year_based | validate_2025 | Logistic Regression
Running year_based | validate_2025 | Random Forest
Running year_based | validate_2025 | XGBoost
Running year_based | validate_2025 | CatBoost
Running expanding_window | validate_2024_H1 | Logistic Regression
Running expanding_window | validate_2024_H1 | Random Forest
Running expanding_window | validate_2024_H1 | XGBoost
Running expanding_window | validate_2024_H1 | CatBoost
Running expanding_window | validate_2024_H2 | Logistic Regression
Running expanding_window | validate_2024_H2 | Random Forest
Running expanding_window | validate_2024_H2 | XGBoost
Running expanding_window | validate_2024_H2 | CatBoost
Running expanding_window | validate_2025_H1 | Logistic Regression
Running expanding_window | validate_2025_H1 | Random Forest
Runnin

c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['team_a_elo' 'team_b_elo' 'elo_diff']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['team_a_elo' 'team_b_elo' 'elo_diff']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['team_a_elo' 'team_b_elo' 'elo_diff']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any ob

Running rolling_5_year_window | validate_2024_H1 | XGBoost


c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['team_a_elo' 'team_b_elo' 'elo_diff']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['team_a_elo' 'team_b_elo' 'elo_diff']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['team_a_elo' 'team_b_elo' 'elo_diff']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any ob

Running rolling_5_year_window | validate_2024_H1 | CatBoost


...

c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['team_a_elo' 'team_b_elo' 'elo_diff']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['team_a_elo' 'team_b_elo' 'elo_diff']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['team_a_elo' 'team_b_elo' 'elo_diff']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any ob

Running rolling_5_year_window | validate_2025_H2 | CatBoost


c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['team_a_elo' 'team_b_elo' 'elo_diff']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['team_a_elo' 'team_b_elo' 'elo_diff']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['team_a_elo' 'team_b_elo' 'elo_diff']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\SAMARTH\anaconda3\envs\football_ml\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any ob

,experiment,cv_method,fold,train_start,train_end,valid_start,valid_end,train_rows,fit_rows,calibration_rows,valid_rows,model,calibration,n_matches,accuracy,macro_f1,log_loss,brier,rps
0,cv_2024_2025,year_based,validate_2024,None,2023-12-31,2024-01-01,2024-12-31,46861,37489,9372,1231,Majority baseline,none,1231,0.461413,0.210487,1.064086,0.642693,0.196656
1,cv_2024_2025,year_based,validate_2024,None,2023-12-31,2024-01-01,2024-12-31,46861,37489,9372,1231,Logistic Regression,none,1231,0.571893,0.512891,0.907177,0.536040,0.172235
2,cv_2024_2025,year_based,validate_2024,None,2023-12-31,2024-01-01,2024-12-31,46861,37489,9372,1231,Logistic Regression,sigmoid,1231,0.587327,0.435090,0.901094,0.531121,0.171953
3,cv_2024_2025,year_based,validate_2024,None,2023-12-31,2024-01-01,2024-12-31,46861,37489,9372,1231,Logistic Regression,isotonic,1231,0.586515,0.435434,0.928075,0.531906,0.172252
4,cv_2024_2025,year_based,validate_2024,None,2023-12-31,2024-01-01,2024-12-31,46861,37489,9372,1231,Random Forest,none,1231,0.571080,0.517541,0.917488,0.543917,0.173571


Total result rows: 130


In [77]:
metric_cols = ["accuracy", "macro_f1", "log_loss", "brier", "rps"]

cv_summary_df = (
    cv_results_df
    .groupby(["cv_method", "model", "calibration"], as_index=False)
    .agg(
        folds=("fold", "nunique"),
        avg_accuracy=("accuracy", "mean"),
        avg_macro_f1=("macro_f1", "mean"),
        avg_log_loss=("log_loss", "mean"),
        avg_brier=("brier", "mean"),
        avg_rps=("rps", "mean"),
        total_matches=("n_matches", "sum"),
    )
    .sort_values(
        ["avg_rps", "avg_log_loss", "avg_brier", "avg_accuracy"],
        ascending=[True, True, True, False]
    )
    .reset_index(drop=True)
)

display(cv_summary_df)

,cv_method,model,calibration,folds,avg_accuracy,avg_macro_f1,avg_log_loss,avg_brier,avg_rps,total_matches
0,year_based,CatBoost,isotonic,2,0.612340,0.462695,0.857137,0.504142,0.161728,2233
1,expanding_window,CatBoost,isotonic,4,0.609962,0.460243,0.855416,0.503734,0.161746,2233
2,expanding_window,XGBoost,isotonic,4,0.611952,0.457062,0.869852,0.504013,0.161852,2233
3,year_based,Random Forest,isotonic,2,0.609033,0.451015,0.857878,0.504779,0.161880,2233
4,expanding_window,Random Forest,isotonic,4,0.609034,0.450002,0.858027,0.505168,0.161975,2233
5,year_based,CatBoost,none,2,0.607223,0.447720,0.861149,0.506368,0.162069,2233
6,year_based,XGBoost,isotonic,2,0.608163,0.456474,0.884692,0.505372,0.162090,2233
7,expanding_window,CatBoost,none,4,0.607395,0.447183,0.860681,0.506304,0.162234,2233
8,rolling_5_year_window,CatBoost,none,4,0.603551,0.464534,0.857068,0.504483,0.162399,2233
9,expanding_window,XGBoost,none,4,0.608314,0.449705,0.862917,0.506959,0.162423,2233


In [78]:
non_cv_2026_results = []

train_final_df = get_period(model_df, end="2025-12-31")
test_2026_df = get_period(model_df, start="2026-01-01")

print("Final non-CV training rows:", len(train_final_df))
print("Final 2026 test rows:", len(test_2026_df))

# Majority baseline
non_cv_2026_results.append(
    evaluate_majority_baseline(
        train_final_df,
        test_2026_df,
        metadata={
            "experiment": "final_2026_non_cv",
            "cv_method": "none",
            "fold": "final_2026",
            "train_start": None,
            "train_end": "2025-12-31",
            "valid_start": "2026-01-01",
            "valid_end": str(test_2026_df[DATE_COL].max().date()),
            "train_rows": len(train_final_df),
            "fit_rows": len(train_final_df),
            "calibration_rows": 0,
            "valid_rows": len(test_2026_df),
        }
    )
)

X_train_final, y_train_final = get_X_y(train_final_df)

for model_name, base_estimator in model_registry.items():
    print(f"Final non-CV 2026 test | {model_name}")

    fitted_model = clone(base_estimator)
    fitted_model.fit(X_train_final, y_train_final)

    non_cv_2026_results.append(
        evaluate_model_on_df(
            fitted_model,
            test_2026_df,
            metadata={
                "experiment": "final_2026_non_cv",
                "cv_method": "none",
                "fold": "final_2026",
                "train_start": None,
                "train_end": "2025-12-31",
                "valid_start": "2026-01-01",
                "valid_end": str(test_2026_df[DATE_COL].max().date()),
                "train_rows": len(train_final_df),
                "fit_rows": len(train_final_df),
                "calibration_rows": 0,
                "valid_rows": len(test_2026_df),
                "model": model_name,
                "calibration": "none",
            }
        )
    )


non_cv_2026_results_df = pd.DataFrame(non_cv_2026_results)

display(
    non_cv_2026_results_df.sort_values(
        ["rps", "log_loss", "brier", "accuracy"],
        ascending=[True, True, True, False]
    )
)

Final non-CV training rows: 49094
Final 2026 test rows: 359
Final non-CV 2026 test | Logistic Regression
Final non-CV 2026 test | Random Forest
Final non-CV 2026 test | XGBoost
Final non-CV 2026 test | CatBoost


,experiment,cv_method,fold,train_start,train_end,valid_start,valid_end,train_rows,fit_rows,calibration_rows,valid_rows,model,calibration,n_matches,accuracy,macro_f1,log_loss,brier,rps
4,final_2026_non_cv,none,final_2026,None,2025-12-31,2026-01-01,2026-06-23,49094,49094,0,359,CatBoost,none,359,0.579387,0.412933,0.877479,0.518012,0.164058
3,final_2026_non_cv,none,final_2026,None,2025-12-31,2026-01-01,2026-06-23,49094,49094,0,359,XGBoost,none,359,0.576602,0.411102,0.878478,0.518524,0.164620
1,final_2026_non_cv,none,final_2026,None,2025-12-31,2026-01-01,2026-06-23,49094,49094,0,359,Logistic Regression,none,359,0.571031,0.529138,0.913247,0.542893,0.168411
2,final_2026_non_cv,none,final_2026,None,2025-12-31,2026-01-01,2026-06-23,49094,49094,0,359,Random Forest,none,359,0.557103,0.518253,0.925751,0.550071,0.169524
0,final_2026_non_cv,none,final_2026,None,2025-12-31,2026-01-01,2026-06-23,49094,49094,0,359,Majority baseline,none,359,0.498607,0.221809,1.044423,0.627652,0.188790


In [79]:
calibrated_2026_results = []

fit_final_df = get_period(model_df, end="2023-12-31")
calibration_final_df = get_period(model_df, start="2024-01-01", end="2025-12-31")
test_2026_df = get_period(model_df, start="2026-01-01")

print("Final calibrated fit rows <= 2023:", len(fit_final_df))
print("Final calibration rows 2024-2025:", len(calibration_final_df))
print("Final 2026 test rows:", len(test_2026_df))

X_fit_final, y_fit_final = get_X_y(fit_final_df)
X_calibration_final, y_calibration_final = get_X_y(calibration_final_df)

for model_name, base_estimator in model_registry.items():
    print(f"Final calibrated 2026 test | {model_name}")

    fitted_model = clone(base_estimator)
    fitted_model.fit(X_fit_final, y_fit_final)

    # Optional direct comparison: same <=2023 model without calibration
    calibrated_2026_results.append(
        evaluate_model_on_df(
            fitted_model,
            test_2026_df,
            metadata={
                "experiment": "final_2026_calibration_design",
                "cv_method": "none",
                "fold": "final_2026",
                "train_start": None,
                "train_end": "2023-12-31",
                "valid_start": "2026-01-01",
                "valid_end": str(test_2026_df[DATE_COL].max().date()),
                "train_rows": len(fit_final_df),
                "fit_rows": len(fit_final_df),
                "calibration_rows": 0,
                "valid_rows": len(test_2026_df),
                "model": model_name,
                "calibration": "none_train_to_2023",
            }
        )
    )

    for calibration_method in ["sigmoid", "isotonic"]:
        calibrated_model = make_prefit_calibrator(
            fitted_model,
            method=calibration_method
        )

        calibrated_model.fit(X_calibration_final, y_calibration_final)

        calibrated_2026_results.append(
            evaluate_model_on_df(
                calibrated_model,
                test_2026_df,
                metadata={
                    "experiment": "final_2026_calibrated",
                    "cv_method": "calibration_holdout_2024_2025",
                    "fold": "final_2026",
                    "train_start": None,
                    "train_end": "2023-12-31",
                    "valid_start": "2026-01-01",
                    "valid_end": str(test_2026_df[DATE_COL].max().date()),
                    "train_rows": len(fit_final_df) + len(calibration_final_df),
                    "fit_rows": len(fit_final_df),
                    "calibration_rows": len(calibration_final_df),
                    "valid_rows": len(test_2026_df),
                    "model": model_name,
                    "calibration": calibration_method,
                }
            )
        )


calibrated_2026_results_df = pd.DataFrame(calibrated_2026_results)

display(
    calibrated_2026_results_df.sort_values(
        ["rps", "log_loss", "brier", "accuracy"],
        ascending=[True, True, True, False]
    )
)

Final calibrated fit rows <= 2023: 46861
Final calibration rows 2024-2025: 2233
Final 2026 test rows: 359
Final calibrated 2026 test | Logistic Regression
Final calibrated 2026 test | Random Forest
Final calibrated 2026 test | XGBoost
Final calibrated 2026 test | CatBoost


,experiment,cv_method,fold,train_start,train_end,valid_start,valid_end,train_rows,fit_rows,calibration_rows,valid_rows,model,calibration,n_matches,accuracy,macro_f1,log_loss,brier,rps
11,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,CatBoost,isotonic,359,0.582173,0.433765,0.875418,0.517656,0.163033
5,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,Random Forest,isotonic,359,0.576602,0.415879,0.879731,0.521613,0.164014
4,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,Random Forest,sigmoid,359,0.576602,0.464939,0.880394,0.520625,0.164150
8,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,XGBoost,isotonic,359,0.590529,0.452674,0.882896,0.519786,0.164349
9,final_2026_calibration_design,none,final_2026,None,2023-12-31,2026-01-01,2026-06-23,46861,46861,0,359,CatBoost,none_train_to_2023,359,0.576602,0.412275,0.879842,0.519526,0.164376
10,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,CatBoost,sigmoid,359,0.584958,0.468441,0.881706,0.519977,0.164789
6,final_2026_calibration_design,none,final_2026,None,2023-12-31,2026-01-01,2026-06-23,46861,46861,0,359,XGBoost,none_train_to_2023,359,0.579387,0.413258,0.884787,0.521158,0.164988
7,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,XGBoost,sigmoid,359,0.590529,0.479065,0.885144,0.521206,0.165255
1,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,Logistic Regression,sigmoid,359,0.582173,0.419384,0.886902,0.524758,0.165626
2,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,Logistic Regression,isotonic,359,0.571031,0.416960,0.889884,0.525528,0.166679


In [80]:
final_2026_results_df = pd.concat(
    [
        non_cv_2026_results_df,
        calibrated_2026_results_df,
    ],
    ignore_index=True
)

final_2026_results_df = final_2026_results_df.sort_values(
    ["rps", "log_loss", "brier", "accuracy"],
    ascending=[True, True, True, False]
).reset_index(drop=True)

display(final_2026_results_df)

print("Best 2026 result by RPS, log loss, Brier, then accuracy:")
display(final_2026_results_df.head(10))

,experiment,cv_method,fold,train_start,train_end,valid_start,valid_end,train_rows,fit_rows,calibration_rows,valid_rows,model,calibration,n_matches,accuracy,macro_f1,log_loss,brier,rps
0,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,CatBoost,isotonic,359,0.582173,0.433765,0.875418,0.517656,0.163033
1,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,Random Forest,isotonic,359,0.576602,0.415879,0.879731,0.521613,0.164014
2,final_2026_non_cv,none,final_2026,None,2025-12-31,2026-01-01,2026-06-23,49094,49094,0,359,CatBoost,none,359,0.579387,0.412933,0.877479,0.518012,0.164058
3,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,Random Forest,sigmoid,359,0.576602,0.464939,0.880394,0.520625,0.164150
4,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,XGBoost,isotonic,359,0.590529,0.452674,0.882896,0.519786,0.164349
5,final_2026_calibration_design,none,final_2026,None,2023-12-31,2026-01-01,2026-06-23,46861,46861,0,359,CatBoost,none_train_to_2023,359,0.576602,0.412275,0.879842,0.519526,0.164376
6,final_2026_non_cv,none,final_2026,None,2025-12-31,2026-01-01,2026-06-23,49094,49094,0,359,XGBoost,none,359,0.576602,0.411102,0.878478,0.518524,0.164620
7,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,CatBoost,sigmoid,359,0.584958,0.468441,0.881706,0.519977,0.164789
8,final_2026_calibration_design,none,final_2026,None,2023-12-31,2026-01-01,2026-06-23,46861,46861,0,359,XGBoost,none_train_to_2023,359,0.579387,0.413258,0.884787,0.521158,0.164988
9,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,XGBoost,sigmoid,359,0.590529,0.479065,0.885144,0.521206,0.165255


Best 2026 result by RPS, log loss, Brier, then accuracy:


,experiment,cv_method,fold,train_start,train_end,valid_start,valid_end,train_rows,fit_rows,calibration_rows,valid_rows,model,calibration,n_matches,accuracy,macro_f1,log_loss,brier,rps
0,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,CatBoost,isotonic,359,0.582173,0.433765,0.875418,0.517656,0.163033
1,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,Random Forest,isotonic,359,0.576602,0.415879,0.879731,0.521613,0.164014
2,final_2026_non_cv,none,final_2026,None,2025-12-31,2026-01-01,2026-06-23,49094,49094,0,359,CatBoost,none,359,0.579387,0.412933,0.877479,0.518012,0.164058
3,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,Random Forest,sigmoid,359,0.576602,0.464939,0.880394,0.520625,0.164150
4,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,XGBoost,isotonic,359,0.590529,0.452674,0.882896,0.519786,0.164349
5,final_2026_calibration_design,none,final_2026,None,2023-12-31,2026-01-01,2026-06-23,46861,46861,0,359,CatBoost,none_train_to_2023,359,0.576602,0.412275,0.879842,0.519526,0.164376
6,final_2026_non_cv,none,final_2026,None,2025-12-31,2026-01-01,2026-06-23,49094,49094,0,359,XGBoost,none,359,0.576602,0.411102,0.878478,0.518524,0.164620
7,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,CatBoost,sigmoid,359,0.584958,0.468441,0.881706,0.519977,0.164789
8,final_2026_calibration_design,none,final_2026,None,2023-12-31,2026-01-01,2026-06-23,46861,46861,0,359,XGBoost,none_train_to_2023,359,0.579387,0.413258,0.884787,0.521158,0.164988
9,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,XGBoost,sigmoid,359,0.590529,0.479065,0.885144,0.521206,0.165255


In [81]:
best_cv_choice = cv_summary_df.iloc[0]

best_cv_method = best_cv_choice["cv_method"]
best_model = best_cv_choice["model"]
best_calibration = best_cv_choice["calibration"]

print("Best CV-selected choice:")
print("CV method:", best_cv_method)
print("Model:", best_model)
print("Calibration:", best_calibration)

matching_2026_rows = final_2026_results_df[
    (final_2026_results_df["model"] == best_model)
]

if best_calibration == "none":
    matching_2026_rows = matching_2026_rows[
        matching_2026_rows["calibration"].isin(["none", "none_train_to_2023"])
    ]
else:
    matching_2026_rows = matching_2026_rows[
        matching_2026_rows["calibration"] == best_calibration
    ]

print("\n2026 result for CV-selected model/calibration:")
display(matching_2026_rows)

Best CV-selected choice:
CV method: year_based
Model: CatBoost
Calibration: isotonic

2026 result for CV-selected model/calibration:


,experiment,cv_method,fold,train_start,train_end,valid_start,valid_end,train_rows,fit_rows,calibration_rows,valid_rows,model,calibration,n_matches,accuracy,macro_f1,log_loss,brier,rps
0,final_2026_calibrated,calibration_holdout_2024_2025,final_2026,None,2023-12-31,2026-01-01,2026-06-23,49094,46861,2233,359,CatBoost,isotonic,359,0.582173,0.433765,0.875418,0.517656,0.163033
